# Data Cleaning: Creating final_3d_df.csv

This notebook combines raw data from multiple sources to create the final training dataset:
- Housing prices (wide format)
- Personal income (wide format)
- Mortgage rates (time series)
- Affordability metrics (long format)

In [14]:
import pandas as pd
import numpy as np





## Load Raw Data

In [15]:
affordability_df = pd.read_csv('/Users/zachrose/Desktop/Neural Networks/Final/Affordable-housing-final/data/raw/affordability_county.csv')
housing_prices_df = pd.read_csv('/Users/zachrose/Desktop/Neural Networks/Final/Affordable-housing-final/data/raw/avg_housing_prices_county.csv')
mortgage_rates_df = pd.read_csv('/Users/zachrose/Desktop/Neural Networks/Final/Affordable-housing-final/data/raw/Mortgage_Rates.csv')
income_df = pd.read_csv('/Users/zachrose/Desktop/Neural Networks/Final/Affordable-housing-final/data/raw/personal_income_county.csv')

## Transform Housing Prices: Wide to Long Format

In [16]:

housing_prices_df['fips'] = housing_prices_df['StateCodeFIPS'] * 1000 + housing_prices_df['MunicipalCodeFIPS']


price_columns = [col for col in housing_prices_df.columns if col.count('-') == 2]  # Date columns
housing_for_melt = housing_prices_df[['fips'] + price_columns].copy()


housing_long = housing_for_melt.melt(id_vars=['fips'], var_name='date', value_name='avg_housing_price')
housing_long['date'] = pd.to_datetime(housing_long['date'])
housing_long = housing_long.sort_values(['fips', 'date']).reset_index(drop=True)


housing_long.head()

/var/folders/sm/hqp5rc_155q39l83bp03k3w40000gn/T/ipykernel_74137/1571250003.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  housing_prices_df['fips'] = housing_prices_df['StateCodeFIPS'] * 1000 + housing_prices_df['MunicipalCodeFIPS']


,fips,date,avg_housing_price
0,1001,2000-01-31,122058.081559
1,1001,2000-02-29,122088.584666
2,1001,2000-03-31,121888.329099
3,1001,2000-04-30,121812.354810
4,1001,2000-05-31,121810.006714


In [17]:

income_df = income_df.rename(columns={'GeoFIPS': 'fips'})


year_columns = [col for col in income_df.columns if col.isdigit()]
income_for_melt = income_df[['fips'] + year_columns].copy()


income_long = income_for_melt.melt(id_vars=['fips'], var_name='year', value_name='income')
income_long['year'] = income_long['year'].astype(int)
income_long = income_long.sort_values(['fips', 'year']).reset_index(drop=True)


income_long.head()

,fips,year,income
0,1001,2000,23584
1,1001,2001,24643
2,1001,2002,25082
3,1001,2003,26500
4,1001,2004,27745


In [18]:

mortgage_rates_df['observation_date'] = pd.to_datetime(mortgage_rates_df['observation_date'])


mortgage_rates_df['year_month'] = mortgage_rates_df['observation_date'].dt.to_period('M')
mortgage_monthly = mortgage_rates_df.groupby('year_month')['MORTGAGE30US'].mean().reset_index()
mortgage_monthly.columns = ['year_month', 'mortgage_rate']
mortgage_monthly['date'] = mortgage_monthly['year_month'].dt.to_timestamp()
mortgage_monthly = mortgage_monthly[['date', 'mortgage_rate']]


mortgage_monthly.head()

,date,mortgage_rate
0,1971-04-01,7.3100
1,1971-05-01,7.4250
2,1971-06-01,7.5300
3,1971-07-01,7.6040
4,1971-08-01,7.6975


In [19]:

affordability_df['date'] = pd.to_datetime(affordability_df['Month'])
affordability_df = affordability_df.rename(columns={'County FIP': 'fips'})


afford_for_merge = affordability_df[['fips', 'date', 'Affordability Metric', 'county_perc_mo']].copy()
afford_for_merge = afford_for_merge.sort_values(['fips', 'date']).reset_index(drop=True)


afford_for_merge.head()

,fips,date,Affordability Metric,county_perc_mo
0,1001,2006-12-01,87.701976,0.342102
1,1001,2007-01-01,91.503678,0.327888
2,1001,2007-02-01,95.883109,0.312912
3,1001,2007-03-01,97.340268,0.308228
4,1001,2007-04-01,108.446069,0.276663


In [20]:

add_mortgage = afford_for_merge.merge(mortgage_monthly, on='date', how='inner')
add_mortgage['year'] = add_mortgage['date'].dt.year

add_mortgage.head()


,fips,date,Affordability Metric,county_perc_mo,mortgage_rate,year
0,1001,2006-12-01,87.701976,0.342102,6.1350,2006
1,1001,2007-01-01,91.503678,0.327888,6.2175,2007
2,1001,2007-02-01,95.883109,0.312912,6.2850,2007
3,1001,2007-03-01,97.340268,0.308228,6.1560,2007
4,1001,2007-04-01,108.446069,0.276663,6.1800,2007


In [21]:
final_df = add_mortgage.merge(income_long, on=['fips', 'year'], how='inner')
final_df.head()

,fips,date,Affordability Metric,county_perc_mo,mortgage_rate,year,income
0,1001,2006-12-01,87.701976,0.342102,6.1350,2006,29942
1,1001,2007-01-01,91.503678,0.327888,6.2175,2007,31665
2,1001,2007-02-01,95.883109,0.312912,6.2850,2007,31665
3,1001,2007-03-01,97.340268,0.308228,6.1560,2007,31665
4,1001,2007-04-01,108.446069,0.276663,6.1800,2007,31665


In [22]:
final_df['date'] = pd.to_datetime(final_df['date'])
final_df['year'] = final_df['date'].dt.year
final_df['month'] = final_df['date'].dt.month
final_df = final_df.drop(columns=['date'])

final_df.head()

,fips,Affordability Metric,county_perc_mo,mortgage_rate,year,income,month
0,1001,87.701976,0.342102,6.1350,2006,29942,12
1,1001,91.503678,0.327888,6.2175,2007,31665,1
2,1001,95.883109,0.312912,6.2850,2007,31665,2
3,1001,97.340268,0.308228,6.1560,2007,31665,3
4,1001,108.446069,0.276663,6.1800,2007,31665,4


In [23]:
# Save final_df to processed data folder
final_df.to_csv('/Users/zachrose/Desktop/Neural Networks/Final/Affordable-housing-final/data/processed/final_3d_df.csv', index=False)